# Hate Speech Classification with Local LLMs (M3 Optimized)

This notebook is designed to run locally on Apple Silicon (M3) using [Ollama](https://ollama.com) for efficient inference.

## Prerequisites & Setup

Before running this notebook, ensure you have the following set up:

1.  **Install Ollama**: Download and install from [ollama.com/download](https://ollama.com/download).
2.  **Pull the Model**: Open your terminal and run:
    ```bash
    ollama pull phi3:mini
    ```
    *Note: We use `phi3:mini` (3.8B parameters) as it is highly capable and runs very fast on M-series chips.*
3.  **Python Environment**: Ensure you are running this in a virtual environment (`.venv`) with the required packages installed:
    ```bash
    python3 -m venv .venv
    source .venv/bin/activate
    pip install openai datasets scikit-learn pandas python-dotenv tqdm ipykernel
    ```
4.  **Confirm Ollama is Running**: The Ollama app should be open (you can see the llama icon in your menu bar).

In [12]:
import subprocess
import sys
import os
import time
import requests
import pandas as pd
import json
from tqdm import tqdm
from openai import OpenAI
from datasets import load_dataset

# --- Configuration ---
MODEL_NAME = "phi3:mini"
OLLAMA_API_URL = "http://localhost:11434"
CHECKPOINT_INTERVAL = 10  # Save every 10 examples
TEST_SUBSET_SIZE = 100 # Number of examples to process for this run

## 1. System Health Check
We verify that Ollama is accessible and the requested model is available locally.

In [13]:
def check_ollama_status(model_name):
    """Verifies Ollama is running and the model is available."""
    try:
        # Check if server is up
        response = requests.get(f"{OLLAMA_API_URL}/api/tags")
        if response.status_code != 200:
            print("❌ Ollama server is not responding. Please start the Ollama app.")
            return False
        
        # Check if model is pulled
        models = [m['name'] for m in response.json()['models']]
        # Handle cases where model tags might have ':latest' implied or explicit
        available = any(model_name in m for m in models)
        
        if not available:
            print(f"⚠️ Model '{model_name}' not found locally.")
            print(f"Please run: ollama pull {model_name}")
            # Optional: Attempt to pull automatically (can take time)
            # subprocess.run(["ollama", "pull", model_name])
            return False
            
        print(f"✅ Ollama is running and '{model_name}' is ready.")
        return True

    except requests.exceptions.ConnectionError:
        print("❌ Could not connect to Ollama. Make sure the app is running (http://localhost:11434).")
        return False

if not check_ollama_status(MODEL_NAME):
    print("Stopping execution due to missing prerequisites.")
    # Stop execution if setup fails
    # sys.exit(1) 

✅ Ollama is running and 'phi3:mini' is ready.


## 2. Load Dataset
We use the HateXplain dataset, normalizing it to extract the text and ground truth labels.

In [14]:
# Data Processing Helpers
ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

# Minimal normalization required to get text and label
def simple_normalize(e):
    # Helper to resolve lists or weird nesting in the dataset
    def get_text(ex):
        if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
            return " ".join(ex["post_tokens"])
        return str(ex.get("text", ""))
    
    def get_label(ex):
        # Local JSONL format: annotators is a dict with a 'label' list
        # e.g., {'label': [1, 1, 1], ...}
        anns = ex.get("annotators", {})
        labels = anns.get("label", []) if isinstance(anns, dict) else []
        
        if not labels: return 1 # Default to normal
        
        counts = {0:0, 1:0, 2:0}
        for lab in labels:
            lab = int(lab)
            if lab in counts: counts[lab] += 1
        
        # Prioritize hate (0) and offensive (2) over normal (1) for tie-breaking usually,
        # but here we just take simple max
        return max(counts, key=counts.get)

    return {
        "text_norm": get_text(e),
        "label_id": get_label(e),
        "label_name": ID2NAME[get_label(e)]
    }

print("Loading Dataset from Local Files...")
try:
    # Load from local JSONL files
    data_files = {
        "train": "hatexplain/hatexplain_train.jsonl",
        "validation": "hatexplain/hatexplain_validation.jsonl",
        "test": "hatexplain/hatexplain_test.jsonl"
    }
    
    # Check if files exist
    for split, path in data_files.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Could not find {split} file at {path}")

    ds = load_dataset("json", data_files=data_files)
    ds = ds.map(simple_normalize)
    
    # Filter out empty text
    ds = ds.filter(lambda x: len(x["text_norm"].strip()) > 0)
    
    # Optional: Filter out 'offensive' if we only want binary hate vs normal
    ds = ds.filter(lambda x: x["label_name"] != "offensive")
    
    # Select subset for testing
    test_ds = ds["test"].select(range(min(len(ds["test"]), TEST_SUBSET_SIZE)))
    print(f"✅ Dataset loaded. Processing {len(test_ds)} examples.")
    
except Exception as e:
    print(f"❌ Error loading dataset: {e}")

Loading Dataset from Local Files...
✅ Dataset loaded. Processing 100 examples.


## 3. Robust Inference Client
We configure an OpenAI-compatible client pointing to our local Ollama instance.

In [16]:
# Initialize Client
client = OpenAI(
    base_url=f"{OLLAMA_API_URL}/v1",
    api_key="ollama" # Required but ignored
)

def classify_text_robust(text, model=MODEL_NAME, retries=3):
    """Classifies text with retry logic and error handling."""
    system_prompt = (
        "You are an expert content moderator. "
        "Classify the following text into one of these two categories: 'hatespeech', 'normal'.\n\n"
        "Definitions:\n"
        "- 'hatespeech': Content that promotes violence, discrimination, or disparagement against protected groups (race, religion, gender, etc.). It aims to incite hatred.\n"
        "Provide ONLY the category name as your response, in lowercase."
    )
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Text: {text}"}
                ],
                temperature=0, # Deterministic
                max_tokens=10,
                timeout=30 # Don't hang forever
            )
            
            prediction = response.choices[0].message.content.strip().lower()
            
            # Clean up response (sometimes models add punctuation)
            if "hatespeech" in prediction: return "hatespeech"
            if "normal" in prediction: return "normal"
            
            # If ambiguous but not empty, return as is (or map to unknown)
            return prediction if prediction else "error_empty"

        except Exception as e:
            if attempt == retries - 1:
                print(f"\n❌ Failed to classify after {retries} attempts: {e}")
                return "api_error"
            time.sleep(1) # Wait a bit before retry

# Warm-up check
print("Testing model connection...")
dummy_pred = classify_text_robust("fucker nigger")
print(f"Warm-up prediction: {dummy_pred}")

Testing model connection...
Warm-up prediction: hatespeech


## 4. Execution with Checkpointing
We process the dataset and save partial results to CSV. This ensures that if the script crashes or you stop it, you don't lose progress.

In [17]:
results = []
checkpoint_path = OUTPUT_CSV

# Create or append? For safety, let's just append to a list and overwrite the file periodically.
# If resuming, you might want to load existing CSV here. 
# For this robust script, we will simply process what we have.

print(f"Starting inference on {len(test_ds)} items...")
print(f"Saving checkpoints to: {os.path.abspath(checkpoint_path)}")

for i, example in enumerate(tqdm(test_ds)):
    text = example['text_norm']
    ground_truth = example['label_name']
    
    prediction = classify_text_robust(text)
    
    results.append({
        "id": i,
        "text": text,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "match": ground_truth == prediction
    })
    
    # Checkpoint
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)

# Final Save
pd.DataFrame(results).to_csv(checkpoint_path, index=False)
print("\n✅ Processing complete. Results saved.")

Starting inference on 100 items...
Saving checkpoints to: /Users/shariff/Documents/github/11th-DAP-hate-speech/hate_speech_predictions_local.csv


100%|██████████| 100/100 [00:28<00:00,  3.46it/s]


✅ Processing complete. Results saved.


## 5. Analysis
Quick look at the performance metrics.

In [18]:
from sklearn.metrics import classification_report, confusion_matrix

df = pd.DataFrame(results)

# Filter out errors if any
valid_df = df[~df['prediction'].isin(['api_error', 'error_empty'])]

print("Classification Report:\n")
print(classification_report(valid_df['ground_truth'], valid_df['prediction']))

print("\nConfusion Matrix:\n")
print(confusion_matrix(valid_df['ground_truth'], valid_df['prediction'], labels=["hatespeech", "normal"]))

Classification Report:

              precision    recall  f1-score   support

  hatespeech       0.48      1.00      0.65        44
      normal       1.00      0.14      0.25        56

    accuracy                           0.52       100
   macro avg       0.74      0.57      0.45       100
weighted avg       0.77      0.52      0.42       100


Confusion Matrix:

[[44  0]
 [48  8]]


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime
import csv

df = pd.DataFrame(results)

# Filter out errors if any
valid_df = df[~df['prediction'].isin(['api_error', 'error_empty'])]
error_count = len(df) - len(valid_df)

# Get ground truth and predictions
y_true = valid_df['ground_truth'].tolist()
y_pred = valid_df['prediction'].tolist()

# Get classification report as dict for structured access
report_dict = classification_report(y_true, y_pred, target_names=["hatespeech", "normal"], output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=["hatespeech", "normal"])

# --- Pretty Print Results ---
print("\n" + "═" * 60)
print("  📊 LLM CLASSIFICATION RESULTS")
print("═" * 60)

# Model Info
print(f"\n  Model: {MODEL_NAME}")
print(f"  Samples: {len(valid_df)} valid / {len(df)} total ({error_count} errors)")

# Overall Metrics
print("\n┌" + "─" * 58 + "┐")
print("│" + " OVERALL METRICS".center(58) + "│")
print("├" + "─" * 29 + "┬" + "─" * 28 + "┤")
print(f"│  Accuracy:  {report_dict['accuracy']*100:>6.2f}%" + " " * 12 + "│" + f"  Macro F1:  {report_dict['macro avg']['f1-score']*100:>6.2f}%" + " " * 6 + "│")
print("└" + "─" * 29 + "┴" + "─" * 28 + "┘")

# Per-Class Metrics
print("\n┌" + "─" * 58 + "┐")
print("│" + " PER-CLASS METRICS".center(58) + "│")
print("├" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 13 + "┤")
print("│" + "  Class".center(14) + "│" + "  Precision".center(14) + "│" + "  Recall".center(14) + "│" + "  F1-Score".center(13) + "│")
print("├" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 13 + "┤")
for class_name in ["hatespeech", "normal"]:
    metrics = report_dict[class_name]
    print(f"│  {class_name:<11} │" + f"  {metrics['precision']*100:>6.2f}%" + "    │" + f"  {metrics['recall']*100:>6.2f}%" + "    │" + f"  {metrics['f1-score']*100:>5.2f}%" + "   │")
print("└" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 13 + "┘")

# Confusion Matrix
print("\n┌" + "─" * 38 + "┐")
print("│" + " CONFUSION MATRIX".center(38) + "│")
print("├" + "─" * 38 + "┤")
print("│" + "              Predicted".ljust(38) + "│")
print("│" + "           hatespeech   normal".ljust(38) + "│")
print("├" + "─" * 38 + "┤")
print(f"│  Actual hatespeech │  {cm[0][0]:>5}   │  {cm[0][1]:>5}   │")
print(f"│         normal     │  {cm[1][0]:>5}   │  {cm[1][1]:>5}   │")
print("└" + "─" * 38 + "┘")

# --- Save Results to CSV ---
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(results_dir, "model_scores.csv")

# Prepare result entry
result_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_name": MODEL_NAME,
    "model_type": "zero-shot (llm)", 
    "task": "binary",
    "epochs": "N/A",  # LLM inference, no training
    "batch_size": "N/A",
    "learning_rate": "N/A",
    "max_length": "N/A",
    "accuracy": round(report_dict["accuracy"], 4),
    "macro_f1": round(report_dict["macro avg"]["f1-score"], 4),
    "macro_precision": round(report_dict["macro avg"]["precision"], 4),
    "macro_recall": round(report_dict["macro avg"]["recall"], 4),
    "hatespeech_f1": round(report_dict["hatespeech"]["f1-score"], 4),
    "hatespeech_precision": round(report_dict["hatespeech"]["precision"], 4),
    "hatespeech_recall": round(report_dict["hatespeech"]["recall"], 4),
    "normal_f1": round(report_dict["normal"]["f1-score"], 4),
    "normal_precision": round(report_dict["normal"]["precision"], 4),
    "normal_recall": round(report_dict["normal"]["recall"], 4),
    "cm_hate_as_hate": int(cm[0][0]),
    "cm_hate_as_normal": int(cm[0][1]),
    "cm_normal_as_hate": int(cm[1][0]),
    "cm_normal_as_normal": int(cm[1][1]),
    "test_samples": len(valid_df),
}

# Check if file exists to determine if we need to write headers
file_exists = os.path.exists(results_file)

with open(results_file, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=result_entry.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(result_entry)

print("\n" + "═" * 60)
print(f"  ✅ Results saved to: {results_file}")
print("═" * 60)


════════════════════════════════════════════════════════════
  📊 LLM CLASSIFICATION RESULTS
════════════════════════════════════════════════════════════

  Model: phi3:mini
  Samples: 100 valid / 100 total (0 errors)

┌──────────────────────────────────────────────────────────┐
│                      OVERALL METRICS                     │
├─────────────────────────────┬────────────────────────────┤
│  Accuracy:   52.00%            │  Macro F1:   44.85%      │
└─────────────────────────────┴────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│                     PER-CLASS METRICS                    │
├──────────────┬──────────────┬──────────────┬─────────────┤
│     Class    │   Precision  │     Recall   │    F1-Score │
├──────────────┼──────────────┼──────────────┼─────────────┤
│  hatespeech  │   47.83%    │  100.00%    │  64.71%   │
│  normal      │  100.00%    │   14.29%    │  25.00%   │
└──────────────┴──────────────┴──────────────┴─────────